# 🦀 Day 7: Mini-Project — Safe Wrapper Around a C Library
---

Today we build an **ergonomic Rust API** over C-style functions.

- **Simulated C API**: Key-value store (`kv_create`, `kv_set`, `kv_get`, `kv_delete`, `kv_destroy`)
- **Raw bindings**: `extern "C"` declarations
- **Safe wrapper**: `KvStore` struct with RAII (`Drop`)
- **Error handling**: C error codes → `Result<T, E>`
- **String conversion**: `CString`/`CStr` ↔ `String`/`&str`

In [ ]:
// Simulated C library: we implement it in Rust to mimic a C library
use std::collections::HashMap;
use std::ffi::{c_int, c_char, c_void, CString, CStr};
use std::ptr;

static mut STORE: Option<HashMap<String, String>> = None;

#[no_mangle]
pub extern "C" fn kv_create() -> *mut c_void {
    unsafe {
        STORE = Some(HashMap::new());
        &mut STORE as *mut _ as *mut c_void
    }
}

#[no_mangle]
pub extern "C" fn kv_set(_: *mut c_void, key: *const c_char, value: *const c_char) -> c_int {
    unsafe {
        if let Some(ref mut m) = STORE {
            let k = CStr::from_ptr(key).to_str().unwrap_or("").to_string();
            let v = CStr::from_ptr(value).to_str().unwrap_or("").to_string();
            m.insert(k, v);
            0
        } else { -1 }
    }
}

#[no_mangle]
pub extern "C" fn kv_get(_: *mut c_void, key: *const c_char) -> *mut c_char {
    unsafe {
        if let Some(ref m) = STORE {
            let k = CStr::from_ptr(key).to_str().unwrap_or("");
            if let Some(v) = m.get(k) {
                CString::new(v.as_str()).unwrap().into_raw()
            } else { ptr::null_mut() }
        } else { ptr::null_mut() }
    }
}

#[no_mangle]
pub extern "C" fn kv_destroy(_: *mut c_void) {
    unsafe { STORE = None; }
}

In [ ]:
// Safe wrapper: KvStore with RAII
pub struct KvStore {
    handle: *mut c_void,
}

impl KvStore {
    pub fn new() -> Result<Self, &'static str> {
        let h = unsafe { kv_create() };
        if h.is_null() { Err("kv_create failed") }
        else { Ok(KvStore { handle: h }) }
    }

    pub fn set(&self, key: &str, value: &str) -> Result<(), &'static str> {
        let k = CString::new(key).map_err(|_| "invalid key")?;
        let v = CString::new(value).map_err(|_| "invalid value")?;
        let ret = unsafe { kv_set(self.handle, k.as_ptr(), v.as_ptr()) };
        if ret == 0 { Ok(()) } else { Err("kv_set failed") }
    }

    pub fn get(&self, key: &str) -> Option<String> {
        let k = CString::new(key).ok()?;
        let ptr = unsafe { kv_get(self.handle, k.as_ptr()) };
        if ptr.is_null() { return None; }
        unsafe {
            let cstr = CString::from_raw(ptr);
            cstr.to_str().ok().map(|s| s.to_string())
        }
    }
}

impl Drop for KvStore {
    fn drop(&mut self) {
        if !self.handle.is_null() {
            unsafe { kv_destroy(self.handle); }
        }
    }
}

In [ ]:
// Testing the safe API
let store = KvStore::new().expect("create failed");
store.set("name", "Rust").expect("set failed");
store.set("version", "1.0").expect("set failed");
println!("get name: {:?}", store.get("name"));
println!("get version: {:?}", store.get("version"));
println!("get missing: {:?}", store.get("missing"));
// Drop is automatic when store goes out of scope

## 📝 Exercise: Add iteration support or serialization

Extend the KvStore safe wrapper with one of:
- **Iteration**: A method that returns all (key, value) pairs
- **Serialization**: A method to export the store as JSON (use serde_json if available)

Or design the C API for `kv_iter_next` that returns the next key-value pair.

In [ ]:
// YOUR CODE HERE

## 🎯 Key Takeaways

1. **Simulated C API**: Define `extern "C"` functions in Rust to mimic a C library for demos
2. **Safe wrapper**: `KvStore` struct with RAII — `Drop` ensures `kv_destroy` is called
3. **Error handling**: `Result<T, E>` for C error codes; `Option<T>` for null returns
4. **String conversion**: `CString`/`CStr` for FFI string boundaries; `CString::from_raw` to reclaim ownership
5. **Memory ownership**: Document who allocates and who frees in your C API

---
**Week 19 Complete! Next week: Performance & Optimization!**